# Engine Training Notebook

Automated pipeline:

1. Read PGNs from `data/lichess_games_export.pgn`
2. Replay engine-side moves against local UCI analysis
3. Build structured move-level diagnostics
4. Train a tabular ML model to predict move quality loss
5. Convert model signals into concrete `Chess/Engine.pm` constant updates
6. Optionally patch `Chess/Engine.pm` directly (no LLM dependency)

## Prerequisites

- `scripts/data_ingress.sh OWN-URLS` has been run.
- Python packages installed from `requirements.txt`.
- Local UCI engine works via `perl play.pl --uci`.

This notebook does not call external APIs.

In [ ]:
from __future__ import annotations

import atexit
import concurrent.futures
import difflib
import io
import json
import multiprocessing
import os
import re
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import chess
import chess.pgn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split


In [ ]:
def _env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def _env_int(name: str, default: int) -> int:
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return int(default)
    try:
        return int(raw.strip())
    except Exception:
        return int(default)


def _env_float(name: str, default: float) -> float:
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return float(default)
    try:
        return float(raw.strip())
    except Exception:
        return float(default)


def _env_list(name: str, default: List[str]) -> List[str]:
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return list(default)
    try:
        if raw.strip().startswith("["):
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    return [part.strip() for part in raw.split(",") if part.strip()]


def _resolve_path(raw: Optional[str], default_path: Path) -> Path:
    if raw is None or not raw.strip():
        return default_path
    parsed = Path(raw.strip())
    if parsed.is_absolute():
        return parsed
    return (Path.cwd().resolve() / parsed).resolve()


REPO_ROOT = Path.cwd().resolve()  # Repository root used to resolve all relative paths.
PGN_PATH = _resolve_path(os.getenv("ENGINE_TRAINING_PGN_PATH"), REPO_ROOT / "data" / "lichess_games_export.pgn")
ENGINE_PM_PATH = REPO_ROOT / "Chess" / "Engine.pm"  # Target engine file for optional auto-patching.
MIGRATION_ROOT = REPO_ROOT / "engineMigration"  # Root directory for migration bundles.
MIGRATION_SUFFIX = os.getenv("ENGINE_TRAINING_MIGRATION_SUFFIX", "engine_training_recommendations")
MIGRATION_TIMESTAMP = os.getenv("ENGINE_TRAINING_MIGRATION_TIMESTAMP") or datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
MIGRATION_DIR = MIGRATION_ROOT / f"V{MIGRATION_TIMESTAMP}__{MIGRATION_SUFFIX}"  # Bundle directory.
PATCH_PREVIEW_PATH = MIGRATION_DIR / "001_engine_patch.diff"  # Ordered migration patch artifact.
MODEL_REPORT_PATH = MIGRATION_DIR / "002_model_report.json"  # Ordered migration model artifact.
REPORT_PATH = MIGRATION_DIR / "003_training_report.json"  # Ordered migration run-report artifact.
MIGRATION_CHECK_CMD = f"scripts/apply_engine_migration.sh check {MIGRATION_DIR.name}"  # Verify patch apply state.
MIGRATION_APPLY_CMD = f"scripts/apply_engine_migration.sh apply {MIGRATION_DIR.name}"  # Apply patch via migration helper.
PARAM_REGISTRY_PATH = REPO_ROOT / "analysis" / "engine_param_registry.json"  # Hyperparameter bounds/weights registry.

# Input source mode:
# - "existing_pgn": use PGN_PATH directly
# - "game_url_log": consume data/lichess_game_urls.log via scripts/data_ingress.sh OWN-URLS into PGN_PATH
# - "lichess_db_urls": download one or more Lichess DB URLs and merge into PGN_PATH
INPUT_SOURCE_MODE = os.getenv("ENGINE_TRAINING_INPUT_SOURCE_MODE", "existing_pgn").strip().lower()
GAME_URL_LOG_PATH = _resolve_path(os.getenv("ENGINE_TRAINING_GAME_URL_LOG_PATH"), REPO_ROOT / "data" / "lichess_game_urls.log")
DATA_INGRESS_SCRIPT_PATH = REPO_ROOT / "scripts" / "data_ingress.sh"  # Unified script for URL-log ingestion.
CLEAR_GAME_URL_LOG_AFTER_CONSUME = _env_bool("ENGINE_TRAINING_CLEAR_GAME_URL_LOG", True)

LICHESS_DB_URLS: List[str] = _env_list("ENGINE_TRAINING_LICHESS_DB_URLS", [])
LICHESS_DB_TMP_DIR = _resolve_path(os.getenv("ENGINE_TRAINING_LICHESS_DB_TMP_DIR"), REPO_ROOT / "data" / "tmp")
KEEP_DB_DOWNLOADS = _env_bool("ENGINE_TRAINING_KEEP_DB_DOWNLOADS", False)

ENGINE_CMD = _env_list("ENGINE_TRAINING_ENGINE_CMD", ["perl", "play.pl", "--uci", "--depth", "8"])  # UCI command used for position analysis.
ENGINE_NAME = os.getenv("ENGINE_TRAINING_ENGINE_NAME", "Perl-GigaChess").strip()
ALLOW_GENERIC_PGN_GAMES = _env_bool("ENGINE_TRAINING_ALLOW_GENERIC_PGN_GAMES", False)
GENERIC_ANALYZE_SIDE_MODE = os.getenv("ENGINE_TRAINING_GENERIC_SIDE_MODE", "white").strip().lower()
if GENERIC_ANALYZE_SIDE_MODE not in {"white", "black", "both"}:
    GENERIC_ANALYZE_SIDE_MODE = "white"

MAX_PLY_PER_GAME = _env_int("ENGINE_TRAINING_MAX_PLY_PER_GAME", 20)
MAX_GAMES = _env_int("ENGINE_TRAINING_MAX_GAMES", 0)  # 0 means no game-count cap.
ANALYSIS_WORKERS = max(1, _env_int("ENGINE_TRAINING_ANALYSIS_WORKERS", 1))
OPENING_PLY_LIMIT = _env_int("ENGINE_TRAINING_OPENING_PLY_LIMIT", 14)
SEARCH_MOVETIME_MS = _env_int("ENGINE_TRAINING_SEARCH_MOVETIME_MS", 250)
EVAL_MOVETIME_MS = _env_int("ENGINE_TRAINING_EVAL_MOVETIME_MS", 250)
ANALYZE_PLAYED_EVAL = _env_bool("ENGINE_TRAINING_ANALYZE_PLAYED_EVAL", True)
ENABLE_BOOK_COMPARE = _env_bool("ENGINE_TRAINING_ENABLE_BOOK_COMPARE", True)

CHECKPOINT_EVERY_GAMES = _env_int("ENGINE_TRAINING_CHECKPOINT_EVERY_GAMES", 250)
PROGRESS_EVERY_GAMES = _env_int("ENGINE_TRAINING_PROGRESS_EVERY_GAMES", 25)
WRITE_CHECKPOINT_ROWS = _env_bool("ENGINE_TRAINING_WRITE_CHECKPOINT_ROWS", True)
CHECKPOINT_DIR = MIGRATION_DIR / "checkpoints"
CHECKPOINT_ROWS_PATH = CHECKPOINT_DIR / "001_partial_rows.jsonl"
CHECKPOINT_PROGRESS_PATH = CHECKPOINT_DIR / "002_progress.json"

MIN_ROWS_FOR_MODEL = _env_int("ENGINE_TRAINING_MIN_ROWS_FOR_MODEL", 24)
SEVERE_CP_LOSS = _env_int("ENGINE_TRAINING_SEVERE_CP_LOSS", 150)
HOLDOUT_FRACTION = _env_float("ENGINE_TRAINING_HOLDOUT_FRACTION", 0.25)
RANDOM_SEED = _env_int("ENGINE_TRAINING_RANDOM_SEED", 42)
MIN_PARAM_SCORE_FOR_UPDATE = _env_float("ENGINE_TRAINING_MIN_PARAM_SCORE", 0.12)
MAX_CONSTANT_UPDATES = _env_int("ENGINE_TRAINING_MAX_CONSTANT_UPDATES", 14)
CANDIDATE_SCALES = _env_list("ENGINE_TRAINING_CANDIDATE_SCALES", ["0.6", "1.0", "1.4"])
CANDIDATE_SCALES = [float(x) for x in CANDIDATE_SCALES]
TRAINING_REFERENCE_GAMES = _env_int("ENGINE_TRAINING_REFERENCE_GAMES", 20_000)
TRAINING_VOLUME_EXPONENT = _env_float("ENGINE_TRAINING_VOLUME_EXPONENT", 0.5)

RUN_POST_PATCH_EVAL = _env_bool("ENGINE_TRAINING_RUN_POST_PATCH_EVAL", False)
POST_PATCH_MAX_GAMES = _env_int("ENGINE_TRAINING_POST_PATCH_MAX_GAMES", MAX_GAMES)
APPLY_ENGINE_PATCH = _env_bool("ENGINE_TRAINING_APPLY_ENGINE_PATCH", False)
WRITE_PATCH_PREVIEW = _env_bool("ENGINE_TRAINING_WRITE_PATCH_PREVIEW", True)

MIGRATION_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

PIECE_VALUE_CP = {
    chess.PAWN: 100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK: 500,
    chess.QUEEN: 900,
    chess.KING: 0,
}

print(f"Repo root: {REPO_ROOT}")
print(f"Source mode: {INPUT_SOURCE_MODE}")
print(f"PGN path: {PGN_PATH}")
print(f"Engine: {' '.join(ENGINE_CMD)}")
print(f"Migration bundle: {MIGRATION_DIR}")
print(f"Registry: {PARAM_REGISTRY_PATH}")
print(f"Generic PGN mode: {ALLOW_GENERIC_PGN_GAMES}")
print(f"Generic side mode: {GENERIC_ANALYZE_SIDE_MODE}")
print(f"Max ply per game: {MAX_PLY_PER_GAME}")
print(f"Analysis workers: {ANALYSIS_WORKERS}")
print(f"Checkpoint every games: {CHECKPOINT_EVERY_GAMES}")
print(f"Progress logging every games: {PROGRESS_EVERY_GAMES}")
print(f"Checkpoint progress file: {CHECKPOINT_PROGRESS_PATH}")




In [ ]:
@dataclass
class UciResult:
    bestmove: Optional[str]
    score_cp: Optional[int]
    depth: Optional[int]
    raw_lines: List[str]


class UciEngineSession:
    def __init__(self, cmd: List[str], cwd: Path):
        self.proc = subprocess.Popen(
            cmd,
            cwd=str(cwd),
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
            bufsize=1,
        )
        self._init_uci()

    def _send(self, command: str) -> None:
        assert self.proc.stdin is not None
        self.proc.stdin.write(command + "\n")
        self.proc.stdin.flush()

    def _read_until_prefix(self, prefix: str) -> List[str]:
        assert self.proc.stdout is not None
        lines: List[str] = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                raise RuntimeError("UCI process closed unexpectedly")
            line = line.rstrip("\n")
            lines.append(line)
            if line.startswith(prefix):
                return lines

    def _init_uci(self) -> None:
        self._send("uci")
        self._read_until_prefix("uciok")
        self._send("setoption name MoveOverhead value 0")
        self._send("isready")
        self._read_until_prefix("readyok")

    @staticmethod
    def _score_from_info(line: str) -> Optional[int]:
        cp_match = re.search(r"score cp (-?\d+)", line)
        if cp_match:
            return int(cp_match.group(1))

        mate_match = re.search(r"score mate (-?\d+)", line)
        if mate_match:
            mate = int(mate_match.group(1))
            return 900000 if mate > 0 else -900000

        return None

    def query(self, moves_uci: List[str], ownbook: bool, movetime_ms: int) -> UciResult:
        book_val = "true" if ownbook else "false"
        self._send(f"setoption name OwnBook value {book_val}")

        if moves_uci:
            self._send("position startpos moves " + " ".join(moves_uci))
        else:
            self._send("position startpos")

        self._send(f"go movetime {int(movetime_ms)}")
        lines = self._read_until_prefix("bestmove")

        bestmove = None
        score_cp = None
        depth = None
        for line in lines:
            if line.startswith("info depth"):
                d_match = re.search(r"info depth (\d+)", line)
                if d_match:
                    depth = int(d_match.group(1))
                parsed = self._score_from_info(line)
                if parsed is not None:
                    score_cp = parsed
            elif line.startswith("bestmove "):
                parts = line.split()
                bestmove = parts[1] if len(parts) > 1 else None

        return UciResult(bestmove=bestmove, score_cp=score_cp, depth=depth, raw_lines=lines)

    def close(self) -> None:
        if self.proc.poll() is not None:
            return
        try:
            self._send("quit")
        except Exception:
            pass
        self.proc.wait(timeout=3)

In [ ]:
def _nonempty_log_entries(path: Path) -> List[str]:
    if not path.exists():
        return []
    entries: List[str] = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        entries.append(line)
    return entries


def _truncate_file(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("", encoding="utf-8")


def _append_plain_pgn(src: Path, out_handle) -> None:
    with src.open("r", encoding="utf-8", errors="ignore") as handle:
        while True:
            chunk = handle.read(1 << 20)
            if not chunk:
                break
            out_handle.write(chunk)


def _append_zst_pgn(src: Path, out_handle) -> None:
    try:
        import zstandard as zstd
    except ImportError as exc:
        raise RuntimeError("zstandard package is required to decode .zst files") from exc

    with src.open("rb") as compressed:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed) as reader:
            text_reader = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
            while True:
                chunk = text_reader.read(1 << 20)
                if not chunk:
                    break
                out_handle.write(chunk)


def _download_lichess_db_urls(urls: List[str], tmp_dir: Path, merged_pgn: Path, keep_downloads: bool) -> Path:
    if not urls:
        raise ValueError("LICHESS_DB_URLS is empty")

    tmp_dir.mkdir(parents=True, exist_ok=True)
    merged_pgn.parent.mkdir(parents=True, exist_ok=True)

    with merged_pgn.open("w", encoding="utf-8") as out:
        for idx, url in enumerate(urls, start=1):
            base_name = Path(url.split("?")[0]).name or f"lichess_{idx}.pgn"
            dest = tmp_dir / base_name
            cmd = ["curl", "-fL", "--retry", "3", "--retry-delay", "1", url, "-o", str(dest)]
            print("Downloading:", url)
            result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(f"curl failed for {url}: {result.stderr.strip() or result.stdout.strip()}")

            if dest.suffix.lower() == ".zst":
                _append_zst_pgn(dest, out)
            else:
                _append_plain_pgn(dest, out)
            out.write("\n\n")

            if not keep_downloads:
                dest.unlink(missing_ok=True)

    return merged_pgn


def _consume_game_url_log(log_path: Path, out_pgn: Path) -> Path:
    entries = _nonempty_log_entries(log_path)
    if not entries:
        raise ValueError(f"No URLs/game IDs found in {log_path}")
    if not DATA_INGRESS_SCRIPT_PATH.exists():
        raise FileNotFoundError(f"Missing data ingress script: {DATA_INGRESS_SCRIPT_PATH}")

    cmd = [
        str(DATA_INGRESS_SCRIPT_PATH),
        "OWN-URLS",
        "--own-url-log",
        str(log_path),
        "--own-pgn-output",
        str(out_pgn),
    ]
    if CLEAR_GAME_URL_LOG_AFTER_CONSUME:
        cmd.append("--clear-own-url-log")
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())

    if CLEAR_GAME_URL_LOG_AFTER_CONSUME:
        print(f"Cleared consumed URL log: {log_path}")

    return out_pgn


def prepare_input_pgn() -> Path:
    mode = INPUT_SOURCE_MODE.strip().lower()
    if mode == "existing_pgn":
        if not PGN_PATH.exists():
            raise FileNotFoundError(f"Missing PGN file: {PGN_PATH}")
        return PGN_PATH

    if mode == "game_url_log":
        return _consume_game_url_log(GAME_URL_LOG_PATH, PGN_PATH)

    if mode == "lichess_db_urls":
        return _download_lichess_db_urls(LICHESS_DB_URLS, LICHESS_DB_TMP_DIR, PGN_PATH, KEEP_DB_DOWNLOADS)

    raise ValueError(f"Unsupported INPUT_SOURCE_MODE: {INPUT_SOURCE_MODE}")


ACTIVE_PGN_PATH = prepare_input_pgn()
print(f"Using PGN: {ACTIVE_PGN_PATH}")

In [ ]:
CENTER_SQUARES = (chess.D4, chess.E4, chess.D5, chess.E5)
HOME_MINOR_SQUARES = {
    chess.WHITE: (chess.B1, chess.G1, chess.C1, chess.F1),
    chess.BLACK: (chess.B8, chess.G8, chess.C8, chess.F8),
}


def material_eval_cp(board: chess.Board, side: chess.Color) -> int:
    ours = 0
    theirs = 0
    for square, piece in board.piece_map().items():
        value = PIECE_VALUE_CP.get(piece.piece_type, 0)
        if piece.color == side:
            ours += value
        else:
            theirs += value
    return ours - theirs


def non_pawn_material_eval_cp(board: chess.Board, side: chess.Color) -> int:
    ours = 0
    theirs = 0
    for square, piece in board.piece_map().items():
        if piece.piece_type in (chess.PAWN, chess.KING):
            continue
        value = PIECE_VALUE_CP.get(piece.piece_type, 0)
        if piece.color == side:
            ours += value
        else:
            theirs += value
    return ours - theirs


def piece_count_non_king(board: chess.Board, side: chess.Color) -> int:
    count = 0
    for square, piece in board.piece_map().items():
        if piece.color == side and piece.piece_type != chess.KING:
            count += 1
    return count


def king_ring_squares(board: chess.Board, color: chess.Color) -> List[int]:
    king_sq = board.king(color)
    if king_sq is None:
        return []
    ring: List[int] = []
    for sq in chess.SquareSet(chess.BB_KING_ATTACKS[king_sq]):
        ring.append(int(sq))
    return ring


def king_danger_proxy(board: chess.Board, side: chess.Color) -> Dict[str, int]:
    king_sq = board.king(side)
    if king_sq is None:
        return {
            "king_in_check": 0,
            "king_ring_attacked": 0,
            "king_ring_undefended": 0,
        }

    enemy = not side
    ring = king_ring_squares(board, side)
    ring_attacked = 0
    ring_undefended = 0
    for sq in ring:
        if board.is_attacked_by(enemy, sq):
            ring_attacked += 1
            if not board.is_attacked_by(side, sq):
                ring_undefended += 1

    return {
        "king_in_check": int(board.is_check()),
        "king_ring_attacked": int(ring_attacked),
        "king_ring_undefended": int(ring_undefended),
    }


def center_control(board: chess.Board, side: chess.Color) -> int:
    return int(sum(1 for sq in CENTER_SQUARES if board.is_attacked_by(side, sq)))


def hanging_piece_count(board: chess.Board, side: chess.Color) -> int:
    enemy = not side
    count = 0
    for sq, piece in board.piece_map().items():
        if piece.color != side or piece.piece_type == chess.KING:
            continue
        if board.is_attacked_by(enemy, sq) and not board.is_attacked_by(side, sq):
            count += 1
    return count


def king_file_pressure(board: chess.Board, side: chess.Color) -> Dict[str, int]:
    king_sq = board.king(side)
    if king_sq is None:
        return {"king_file_open": 0, "king_adj_files_open": 0}

    own_pawns = board.pieces(chess.PAWN, side)
    king_file = chess.square_file(king_sq)

    has_own_pawn_on_file = any(chess.square_file(sq) == king_file for sq in own_pawns)
    king_file_open = int(not has_own_pawn_on_file)

    adj_open = 0
    for f in (king_file - 1, king_file + 1):
        if f < 0 or f > 7:
            continue
        has_pawn = any(chess.square_file(sq) == f for sq in own_pawns)
        if not has_pawn:
            adj_open += 1

    return {"king_file_open": king_file_open, "king_adj_files_open": int(adj_open)}


def is_castled(board: chess.Board, side: chess.Color) -> int:
    king_sq = board.king(side)
    if king_sq is None:
        return 0
    if side == chess.WHITE:
        return int(king_sq in (chess.G1, chess.C1))
    return int(king_sq in (chess.G8, chess.C8))


def development_lag(board: chess.Board, side: chess.Color) -> int:
    lag = 0
    for sq in HOME_MINOR_SQUARES.get(side, ()): 
        piece = board.piece_at(sq)
        if piece is not None and piece.color == side and piece.piece_type in (chess.KNIGHT, chess.BISHOP):
            lag += 1
    return int(lag)


def phase_flags(board: chess.Board) -> Dict[str, int]:
    non_king = 0
    for sq, piece in board.piece_map().items():
        if piece.piece_type != chess.KING:
            non_king += 1
    return {
        "mid_endgame_phase": int(non_king <= 16),
        "endgame_phase": int(non_king <= 10),
    }


def load_engine_games(path: Path, engine_name: str, max_games: int = 0, allow_generic: bool = False) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing PGN file: {path}")

    use_generic = bool(allow_generic) or not engine_name
    games: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        while True:
            game = chess.pgn.read_game(handle)
            if game is None:
                break

            white = (game.headers.get("White") or "").strip()
            black = (game.headers.get("Black") or "").strip()

            if use_generic:
                games.append(
                    {
                        "headers": dict(game.headers),
                        "engine_is_white": None,
                        "analyze_all_sides": True,
                        "moves": list(game.mainline_moves()),
                    }
                )
            else:
                if white != engine_name and black != engine_name:
                    continue

                engine_is_white = white == engine_name
                games.append(
                    {
                        "headers": dict(game.headers),
                        "engine_is_white": engine_is_white,
                        "analyze_all_sides": False,
                        "moves": list(game.mainline_moves()),
                    }
                )

            if max_games and len(games) >= max_games:
                break

    return games


games = load_engine_games(ACTIVE_PGN_PATH, ENGINE_NAME, MAX_GAMES, allow_generic=ALLOW_GENERIC_PGN_GAMES)
mode_label = "generic(all sides)" if ALLOW_GENERIC_PGN_GAMES or not ENGINE_NAME else f"engine-name({ENGINE_NAME})"
print(f"Loaded {len(games)} game(s) from {ACTIVE_PGN_PATH} using mode={mode_label}")


In [ ]:
def analyze_game(game: Dict[str, Any], engine: UciEngineSession) -> List[Dict[str, Any]]:
    headers = game["headers"]
    moves = game["moves"]
    engine_is_white = game.get("engine_is_white")
    analyze_all_sides = bool(game.get("analyze_all_sides", False))

    board = chess.Board()
    played_prefix: List[str] = []
    rows: List[Dict[str, Any]] = []

    for ply_idx, move in enumerate(moves, start=1):
        if MAX_PLY_PER_GAME and ply_idx > MAX_PLY_PER_GAME:
            break

        if analyze_all_sides:
            if GENERIC_ANALYZE_SIDE_MODE == "both":
                engine_to_move = True
            elif GENERIC_ANALYZE_SIDE_MODE == "black":
                engine_to_move = board.turn == chess.BLACK
            else:
                engine_to_move = board.turn == chess.WHITE
        else:
            engine_to_move = (board.turn == chess.WHITE and engine_is_white) or (board.turn == chess.BLACK and not engine_is_white)

        if engine_to_move:
            engine_side = board.turn
            enemy_side = not engine_side
            played_uci = move.uci()
            move_number = board.fullmove_number
            san = board.san(move)
            fen_before = board.fen()

            no_book = engine.query(played_prefix, ownbook=False, movetime_ms=SEARCH_MOVETIME_MS)
            with_book = None
            if ENABLE_BOOK_COMPARE and ply_idx <= OPENING_PLY_LIMIT:
                with_book = engine.query(played_prefix, ownbook=True, movetime_ms=max(50, SEARCH_MOVETIME_MS // 2))

            cp_best = no_book.score_cp
            cp_played = None
            cp_loss = None

            if ANALYZE_PLAYED_EVAL:
                after_played = engine.query(played_prefix + [played_uci], ownbook=False, movetime_ms=EVAL_MOVETIME_MS)
                if after_played.score_cp is not None:
                    cp_played = -after_played.score_cp

            if cp_best is not None and cp_played is not None:
                cp_loss = cp_best - cp_played

            legal_moves_before = int(board.legal_moves.count())
            board_after = board.copy(stack=False)
            board_after.push(move)
            opp_checks_after = int(sum(1 for m in board_after.legal_moves if board_after.gives_check(m)))
            opponent_legal_after = int(board_after.legal_moves.count())

            kd = king_danger_proxy(board, engine_side)
            king_file = king_file_pressure(board, engine_side)
            phase = phase_flags(board)

            piece_count_ours = piece_count_non_king(board, engine_side)
            piece_count_theirs = piece_count_non_king(board, enemy_side)
            center_ours = center_control(board, engine_side)
            center_theirs = center_control(board, enemy_side)
            threatened_ours = hanging_piece_count(board, engine_side)
            threatened_theirs = hanging_piece_count(board, enemy_side)

            engine_color_label = "white" if engine_side == chess.WHITE else "black"

            rows.append(
                {
                    "site": headers.get("Site"),
                    "date": headers.get("Date"),
                    "result": headers.get("Result"),
                    "engine_color": engine_color_label,
                    "ply": ply_idx,
                    "move_number": move_number,
                    "played_san": san,
                    "played_uci": played_uci,
                    "best_uci_nobook": no_book.bestmove,
                    "best_cp_nobook": cp_best,
                    "played_cp": cp_played,
                    "cp_loss": cp_loss,
                    "opening_phase": int(ply_idx <= OPENING_PLY_LIMIT),
                    "played_is_capture": int(board.is_capture(move)),
                    "played_gives_check": int(board.gives_check(move)),
                    "opponent_checks_after": opp_checks_after,
                    "opponent_legal_moves_after": opponent_legal_after,
                    "book_uci": with_book.bestmove if with_book else None,
                    "book_agrees_played": int(bool(with_book and with_book.bestmove == played_uci)),
                    "book_differs_nobook": int(bool(with_book and with_book.bestmove and no_book.bestmove and with_book.bestmove != no_book.bestmove)),
                    "best_matches_played": int(no_book.bestmove == played_uci),
                    "fen_before": fen_before,
                    "legal_moves": legal_moves_before,
                    "mobility_delta": int(opponent_legal_after - legal_moves_before),
                    "material_cp": int(material_eval_cp(board, engine_side)),
                    "non_pawn_material_cp": int(non_pawn_material_eval_cp(board, engine_side)),
                    "piece_count_ours": int(piece_count_ours),
                    "piece_count_theirs": int(piece_count_theirs),
                    "center_control_ours": int(center_ours),
                    "center_control_theirs": int(center_theirs),
                    "center_control_delta": int(center_ours - center_theirs),
                    "threatened_ours": int(threatened_ours),
                    "threatened_theirs": int(threatened_theirs),
                    "threatened_delta": int(threatened_theirs - threatened_ours),
                    "king_in_check": int(kd["king_in_check"]),
                    "king_ring_attacked": int(kd["king_ring_attacked"]),
                    "king_ring_undefended": int(kd["king_ring_undefended"]),
                    "king_file_open": int(king_file["king_file_open"]),
                    "king_adj_files_open": int(king_file["king_adj_files_open"]),
                    "is_castled": int(is_castled(board, engine_side)),
                    "opening_development_lag": int(development_lag(board, engine_side)),
                    "mid_endgame_phase": int(phase["mid_endgame_phase"]),
                    "endgame_phase": int(phase["endgame_phase"]),
                }
            )

        played_prefix.append(move.uci())
        board.push(move)

    return rows


_WORKER_ENGINE: Optional[UciEngineSession] = None


def _worker_close() -> None:
    global _WORKER_ENGINE
    if _WORKER_ENGINE is not None:
        _WORKER_ENGINE.close()
        _WORKER_ENGINE = None


def _worker_init(engine_cmd: List[str], repo_root: str) -> None:
    global _WORKER_ENGINE
    _WORKER_ENGINE = UciEngineSession(engine_cmd, Path(repo_root))
    atexit.register(_worker_close)


def _worker_analyze_game(task: Tuple[int, Dict[str, Any]]) -> Tuple[int, List[Dict[str, Any]]]:
    game_idx, game = task
    if _WORKER_ENGINE is None:
        raise RuntimeError("Worker engine session is not initialized")
    return game_idx, analyze_game(game, _WORKER_ENGINE)


In [ ]:
all_rows: List[Dict[str, Any]] = []


def _checkpoint_progress(games_done: int, rows_done: int, status: str) -> None:
    payload = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": status,
        "pgn_path": str(ACTIVE_PGN_PATH),
        "migration_dir": str(MIGRATION_DIR),
        "games_total": len(games),
        "games_completed": int(games_done),
        "rows_completed": int(rows_done),
        "checkpoint_every_games": int(CHECKPOINT_EVERY_GAMES),
        "progress_every_games": int(PROGRESS_EVERY_GAMES),
        "write_checkpoint_rows": bool(WRITE_CHECKPOINT_ROWS),
    }
    CHECKPOINT_PROGRESS_PATH.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _emit_progress_and_checkpoint(game_idx: int, game_rows: List[Dict[str, Any]]) -> None:
    all_rows.extend(game_rows)

    if WRITE_CHECKPOINT_ROWS and game_rows:
        with CHECKPOINT_ROWS_PATH.open("a", encoding="utf-8") as handle:
            for row in game_rows:
                handle.write(json.dumps(row, sort_keys=True) + "\n")

    should_checkpoint = False
    if CHECKPOINT_EVERY_GAMES > 0 and game_idx % CHECKPOINT_EVERY_GAMES == 0:
        should_checkpoint = True
    if game_idx == len(games):
        should_checkpoint = True

    should_log_progress = False
    if PROGRESS_EVERY_GAMES > 0 and game_idx % PROGRESS_EVERY_GAMES == 0:
        should_log_progress = True
    if game_idx == len(games):
        should_log_progress = True

    if should_log_progress:
        progress_pct = (100.0 * game_idx / len(games)) if len(games) else 100.0
        print(f"Progress: {game_idx} complete / {len(games)} total games ({progress_pct:.1f}%) rows={len(all_rows)}", flush=True)

    if should_checkpoint:
        _checkpoint_progress(game_idx, len(all_rows), "running")
        print(
            f"Checkpoint: analyzed game {game_idx}/{len(games)} -> {len(game_rows)} engine moves (total_rows={len(all_rows)})",
            flush=True,
        )


if WRITE_CHECKPOINT_ROWS:
    CHECKPOINT_ROWS_PATH.write_text("", encoding="utf-8")

_checkpoint_progress(0, 0, "running")
print(f"Checkpoint rows path: {CHECKPOINT_ROWS_PATH}", flush=True)

games_done = 0
try:
    if ANALYSIS_WORKERS <= 1 or len(games) <= 1:
        engine = UciEngineSession(ENGINE_CMD, REPO_ROOT)
        try:
            for i, game in enumerate(games, start=1):
                rows = analyze_game(game, engine)
                _emit_progress_and_checkpoint(i, rows)
                games_done = i
        finally:
            engine.close()
    else:
        max_workers = min(ANALYSIS_WORKERS, len(games))
        start_method = "fork" if "fork" in multiprocessing.get_all_start_methods() else None
        mp_context = multiprocessing.get_context(start_method) if start_method else None

        print(f"Running parallel game analysis with {max_workers} process worker(s)", flush=True)

        with concurrent.futures.ProcessPoolExecutor(
            max_workers=max_workers,
            initializer=_worker_init,
            initargs=(ENGINE_CMD, str(REPO_ROOT)),
            mp_context=mp_context,
        ) as pool:
            future_to_game: Dict[concurrent.futures.Future, int] = {}
            for i, game in enumerate(games, start=1):
                future = pool.submit(_worker_analyze_game, (i, game))
                future_to_game[future] = i

            pending_results: Dict[int, List[Dict[str, Any]]] = {}
            next_game_idx = 1

            for future in concurrent.futures.as_completed(future_to_game):
                game_idx, rows = future.result()
                pending_results[game_idx] = rows

                while next_game_idx in pending_results:
                    ordered_rows = pending_results.pop(next_game_idx)
                    _emit_progress_and_checkpoint(next_game_idx, ordered_rows)
                    games_done = next_game_idx
                    next_game_idx += 1
except Exception as exc:
    _checkpoint_progress(games_done, len(all_rows), f"error:{type(exc).__name__}")
    raise

_checkpoint_progress(games_done, len(all_rows), "completed")

df = pd.DataFrame(all_rows)
print(f"Total engine moves analyzed: {len(df)}")
if len(df):
    display(df.head(10))





In [ ]:
def summarize_dataset(df: pd.DataFrame) -> Dict[str, Any]:
    if df.empty:
        return {"engine_moves": 0, "games": 0}

    cp_df = df[df["cp_loss"].notna()].copy()
    critical = cp_df[cp_df["cp_loss"] >= 80]
    severe = cp_df[cp_df["cp_loss"] >= SEVERE_CP_LOSS]

    capture_mean = cp_df[cp_df["played_is_capture"] == 1]["cp_loss"].mean() if "played_is_capture" in cp_df.columns else None
    quiet_mean = cp_df[cp_df["played_is_capture"] == 0]["cp_loss"].mean() if "played_is_capture" in cp_df.columns else None
    check_mean = cp_df[cp_df["opponent_checks_after"] > 0]["cp_loss"].mean() if "opponent_checks_after" in cp_df.columns else None
    opening_mean = cp_df[cp_df["opening_phase"] == 1]["cp_loss"].mean() if "opening_phase" in cp_df.columns else None

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "games": int(df["site"].nunique()),
        "engine_moves": int(len(df)),
        "moves_with_eval": int(len(cp_df)),
        "bestmove_match_rate": float(df["best_matches_played"].mean()) if len(df) else None,
        "avg_cp_loss": float(cp_df["cp_loss"].mean()) if len(cp_df) else None,
        "critical_80cp": int(len(critical)),
        "severe_150cp": int(len(severe)),
        "capture_critical": int(len(critical[critical["played_is_capture"] == 1])) if len(critical) else 0,
        "quiet_critical": int(len(critical[critical["played_is_capture"] == 0])) if len(critical) else 0,
        "critical_with_opp_checks": int(len(critical[critical["opponent_checks_after"] > 0])) if len(critical) else 0,
        "book_disagreements_opening": int(df[df["opening_phase"] == 1]["book_differs_nobook"].sum()) if len(df) else 0,
        "capture_avg_cp_loss": float(capture_mean) if capture_mean is not None and pd.notna(capture_mean) else None,
        "quiet_avg_cp_loss": float(quiet_mean) if quiet_mean is not None and pd.notna(quiet_mean) else None,
        "check_pressure_avg_cp_loss": float(check_mean) if check_mean is not None and pd.notna(check_mean) else None,
        "opening_avg_cp_loss": float(opening_mean) if opening_mean is not None and pd.notna(opening_mean) else None,
    }


summary = summarize_dataset(df)
print(json.dumps(summary, indent=2))

if not df.empty and df["cp_loss"].notna().any():
    top = df[df["cp_loss"].notna()].sort_values("cp_loss", ascending=False).head(20)
    display(top[["site", "move_number", "played_san", "played_uci", "best_uci_nobook", "best_cp_nobook", "played_cp", "cp_loss", "played_is_capture", "king_ring_attacked", "opponent_checks_after", "fen_before"]])


## Baseline Visualizations

These charts show **before-change** behavior from the current dataset.

In [ ]:
if df.empty:
    print("No rows to visualize.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    cp = df[df["cp_loss"].notna()]["cp_loss"]
    if len(cp):
        axes[0].hist(cp, bins=20, color="#4472c4", alpha=0.85)
        axes[0].set_title("CP Loss Distribution")
        axes[0].set_xlabel("cp_loss")
        axes[0].set_ylabel("count")
    else:
        axes[0].text(0.5, 0.5, "No cp_loss data", ha="center", va="center")
        axes[0].set_title("CP Loss Distribution")

    by_move = df[df["cp_loss"].notna()].groupby("move_number")["cp_loss"].mean().reset_index()
    if len(by_move):
        axes[1].plot(by_move["move_number"], by_move["cp_loss"], marker="o", color="#70ad47")
        axes[1].set_title("Avg CP Loss by Move Number")
        axes[1].set_xlabel("move_number")
        axes[1].set_ylabel("avg cp_loss")
    else:
        axes[1].text(0.5, 0.5, "No cp_loss data", ha="center", va="center")
        axes[1].set_title("Avg CP Loss by Move Number")

    cap = df[df["cp_loss"].notna()].groupby("played_is_capture")["cp_loss"].mean()
    labels = ["quiet", "capture"]
    vals = [float(cap.get(0, 0.0)), float(cap.get(1, 0.0))]
    axes[2].bar(labels, vals, color=["#ed7d31", "#a5a5a5"])
    axes[2].set_title("Avg CP Loss: Quiet vs Capture")
    axes[2].set_ylabel("avg cp_loss")

    plt.tight_layout()
    plt.show()

if not df.empty:
    risk_df = pd.DataFrame(
        {
            "metric": [
                "bestmove_match_rate",
                "avg_cp_loss",
                "critical_80cp",
                "severe_150cp",
                "book_disagreements_opening",
            ],
            "value": [
                summary.get("bestmove_match_rate"),
                summary.get("avg_cp_loss"),
                summary.get("critical_80cp"),
                summary.get("severe_150cp"),
                summary.get("book_disagreements_opening"),
            ],
        }
    )
    display(risk_df)

## Train ML Model (No LLM)

Model: `GradientBoostingRegressor` on move-level diagnostics to predict `cp_loss`.

Then we use model importances + aggregate risk statistics to propose constant updates.

In [ ]:
feature_cols = [
    "opening_phase",
    "played_is_capture",
    "played_gives_check",
    "opponent_checks_after",
    "opponent_legal_moves_after",
    "book_differs_nobook",
    "best_matches_played",
    "legal_moves",
    "mobility_delta",
    "material_cp",
    "non_pawn_material_cp",
    "piece_count_ours",
    "piece_count_theirs",
    "center_control_delta",
    "threatened_ours",
    "threatened_theirs",
    "threatened_delta",
    "king_in_check",
    "king_ring_attacked",
    "king_ring_undefended",
    "king_file_open",
    "king_adj_files_open",
    "is_castled",
    "opening_development_lag",
    "mid_endgame_phase",
    "endgame_phase",
]

model_metrics: Dict[str, Any] = {}
feature_importance: List[Tuple[str, float]] = []
trained_model = None

if "cp_loss" in df.columns:
    df_eval = df[df["cp_loss"].notna()].copy()
else:
    df_eval = pd.DataFrame(columns=feature_cols + ["cp_loss", "site"])

df_train_eval = df_eval.copy()
df_holdout_eval = df_eval.iloc[0:0].copy()
train_summary_for_optimizer = summarize_dataset(df_train_eval) if not df_train_eval.empty else {"engine_moves": 0, "games": 0}
holdout_summary_for_optimizer = {"engine_moves": 0, "games": 0}

games_requested_count = int(MAX_GAMES) if MAX_GAMES else None
games_loaded_count = int(len(games))
train_game_count = 0
holdout_game_count = 0
training_volume_weight = 0.15

if len(df_eval) < max(10, MIN_ROWS_FOR_MODEL // 2):
    print(f"Not enough labeled rows for robust ML ({len(df_eval)} rows).")

    corr_rows: List[Tuple[str, float]] = []
    for name in feature_cols:
        if name not in df_eval.columns or df_eval[name].nunique(dropna=False) <= 1:
            corr_rows.append((name, 0.0))
            continue
        corr = df_eval[name].astype(float).corr(df_eval["cp_loss"].astype(float))
        corr_rows.append((name, float(abs(corr)) if pd.notna(corr) else 0.0))

    feature_importance = sorted(corr_rows, key=lambda x: x[1], reverse=True)
    train_game_count = int(df_train_eval["site"].nunique()) if "site" in df_train_eval.columns and not df_train_eval.empty else 0
    holdout_game_count = int(df_holdout_eval["site"].nunique()) if "site" in df_holdout_eval.columns and not df_holdout_eval.empty else 0
else:
    split_mode = "row_split"
    unique_games = [x for x in df_eval["site"].dropna().unique().tolist() if str(x)]

    if len(unique_games) >= 4:
        rng = np.random.default_rng(RANDOM_SEED)
        holdout_game_count_target = max(1, int(round(len(unique_games) * HOLDOUT_FRACTION)))
        holdout_set = set(rng.choice(unique_games, size=holdout_game_count_target, replace=False).tolist())

        df_holdout_eval = df_eval[df_eval["site"].isin(holdout_set)].copy()
        df_train_eval = df_eval[~df_eval["site"].isin(holdout_set)].copy()
        split_mode = "game_holdout"

        if df_train_eval.empty or df_holdout_eval.empty:
            df_train_eval, df_holdout_eval = train_test_split(df_eval, test_size=HOLDOUT_FRACTION, random_state=RANDOM_SEED)
            split_mode = "row_split_fallback"
    else:
        df_train_eval, df_holdout_eval = train_test_split(df_eval, test_size=HOLDOUT_FRACTION, random_state=RANDOM_SEED)

    X_train = df_train_eval[feature_cols].fillna(0.0)
    y_train = df_train_eval["cp_loss"].astype(float)
    X_holdout = df_holdout_eval[feature_cols].fillna(0.0)
    y_holdout = df_holdout_eval["cp_loss"].astype(float)

    model = GradientBoostingRegressor(random_state=RANDOM_SEED)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_holdout)
    mae = mean_absolute_error(y_holdout, y_pred)
    r2 = r2_score(y_holdout, y_pred)

    importances = model.feature_importances_
    pairs = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

    trained_model = model
    train_game_count = int(df_train_eval["site"].nunique()) if "site" in df_train_eval.columns else 0
    holdout_game_count = int(df_holdout_eval["site"].nunique()) if "site" in df_holdout_eval.columns else 0

    model_metrics = {
        "rows": int(len(df_eval)),
        "train_rows": int(len(X_train)),
        "holdout_rows": int(len(X_holdout)),
        "mae": float(mae),
        "r2": float(r2),
        "split_mode": split_mode,
        "train_games": train_game_count,
        "holdout_games": holdout_game_count,
    }
    feature_importance = [(name, float(score)) for name, score in pairs]

    print("Model metrics:")
    print(json.dumps(model_metrics, indent=2))

    print("\nFeature importance:")
    for name, score in feature_importance:
        print(f"- {name}: {score:.4f}")

if not model_metrics:
    model_metrics = {
        "rows": int(len(df_eval)),
        "train_rows": int(len(df_train_eval)),
        "holdout_rows": int(len(df_holdout_eval)),
        "mae": None,
        "r2": None,
        "split_mode": "tiny_sample_correlation_fallback",
        "train_games": train_game_count,
        "holdout_games": holdout_game_count,
    }

train_summary_for_optimizer = summarize_dataset(df_train_eval) if not df_train_eval.empty else {"engine_moves": 0, "games": 0}
holdout_summary_for_optimizer = summarize_dataset(df_holdout_eval) if not df_holdout_eval.empty else {"engine_moves": 0, "games": 0}

effective_games_for_weight = train_game_count if train_game_count > 0 else games_loaded_count
training_volume_weight = float(np.clip((max(1, effective_games_for_weight) / max(1, TRAINING_REFERENCE_GAMES)) ** TRAINING_VOLUME_EXPONENT, 0.15, 1.0))

training_counts = {
    "games_requested": games_requested_count,
    "games_loaded": games_loaded_count,
    "games_with_labels": int(len(df_eval)),
    "games_train": int(train_game_count),
    "games_holdout": int(holdout_game_count),
    "training_volume_weight": float(training_volume_weight),
}

model_metrics["training_volume_weight"] = float(training_volume_weight)
print(f"Training volume weight: {training_volume_weight:.3f} (effective games={effective_games_for_weight})")


## Feature Importance Visualization

In [ ]:
if feature_importance:
    fi_df = pd.DataFrame(feature_importance, columns=["feature", "importance"]).sort_values("importance", ascending=True)
    plt.figure(figsize=(8, 5))
    plt.barh(fi_df["feature"], fi_df["importance"], color="#5b9bd5")
    plt.title("Model Feature Importances")
    plt.xlabel("importance")
    plt.tight_layout()
    plt.show()
else:
    print("No model importances available.")

In [ ]:
CONST_PATTERN = re.compile(r"^(\s*use constant\s+([A-Z0-9_]+)\s*=>\s*)([^;]+)(;\s*(?:#.*)?)$")


def parse_engine_constants(path: Path) -> Dict[str, float]:
    constants: Dict[str, float] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        m = CONST_PATTERN.match(line)
        if not m:
            continue
        key = m.group(2)
        raw = m.group(3).strip()
        try:
            constants[key] = float(raw)
        except ValueError:
            continue
    return constants


def _load_notebook_namespace(path: Path) -> Dict[str, object]:
    notebook = json.loads(path.read_text(encoding="utf-8"))
    namespace: Dict[str, object] = {"__name__": "__notebook_loader__", "__file__": str(path)}
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        if not source.strip():
            continue
        exec(compile(source, str(path), "exec"), namespace)
    return namespace

pipeline_ns = _load_notebook_namespace(REPO_ROOT / "analysis" / "engine_hyperparam_pipeline.ipynb")
optimize_constants_from_registry = pipeline_ns["optimize_constants_from_registry"]

engine_constants = parse_engine_constants(ENGINE_PM_PATH)
optimization_payload = optimize_constants_from_registry(
    df_train_eval=df_train_eval,
    df_holdout_eval=df_holdout_eval,
    summary_train=train_summary_for_optimizer,
    summary_holdout=holdout_summary_for_optimizer,
    feature_importance=feature_importance,
    constants=engine_constants,
    registry_path=PARAM_REGISTRY_PATH,
    severe_cp_loss=SEVERE_CP_LOSS,
    candidate_scales=CANDIDATE_SCALES,
    min_param_score=MIN_PARAM_SCORE_FOR_UPDATE,
    max_updates=MAX_CONSTANT_UPDATES,
    training_volume_weight=training_volume_weight,
)

proposed_updates = optimization_payload.get("proposed_updates", {})
selected_candidate = optimization_payload.get("selected_candidate", {})
parameter_rows = optimization_payload.get("parameter_rows", [])
candidate_results = optimization_payload.get("candidate_results", [])
signal_context_train = optimization_payload.get("signal_context_train", {})
signal_context_holdout = optimization_payload.get("signal_context_holdout", {})
group_pressure_train = optimization_payload.get("group_pressure_train", {})
group_pressure_holdout = optimization_payload.get("group_pressure_holdout", {})
optimizer_training_volume_weight = float(optimization_payload.get("training_volume_weight", training_volume_weight))

print("Selected candidate:")
print(json.dumps({
    "name": selected_candidate.get("name"),
    "scale": selected_candidate.get("scale"),
    "proxy_score": selected_candidate.get("proxy_score"),
    "estimated_cp_loss_reduction": selected_candidate.get("estimated_cp_loss_reduction"),
    "changed_constants": selected_candidate.get("changed_constants", []),
    "training_volume_weight": optimizer_training_volume_weight,
}, indent=2))

parameter_weight_df = pd.DataFrame(parameter_rows)
if not parameter_weight_df.empty:
    cols = [
        "name", "group", "current", "direction", "composite_score",
        "group_score", "signal_score", "feature_score", "step", "max_step_change", "training_volume_weight"
    ]
    display(parameter_weight_df[cols].sort_values("composite_score", ascending=False).head(30))

candidate_rows = []
for c in candidate_results:
    candidate_rows.append({
        "candidate": c.get("name"),
        "scale": c.get("scale"),
        "proxy_score": c.get("proxy_score"),
        "estimated_cp_loss_reduction": c.get("estimated_cp_loss_reduction"),
        "changed_count": len(c.get("changed_constants", [])),
        "changed_constants": ", ".join(c.get("changed_constants", [])),
    })
candidate_df = pd.DataFrame(candidate_rows)
if not candidate_df.empty:
    display(candidate_df.sort_values("proxy_score", ascending=False))

updates_df = pd.DataFrame(columns=["constant", "old", "new", "delta", "group", "score"])
if not proposed_updates:
    print("No automatic constant updates proposed from registry optimization.")
else:
    score_map = {row.get("name"): row.get("composite_score") for row in parameter_rows}
    group_map = {row.get("name"): row.get("group") for row in parameter_rows}

    rows = []
    for key, new_value in sorted(proposed_updates.items()):
        old_value = engine_constants.get(key)
        rows.append(
            {
                "constant": key,
                "old": old_value,
                "new": new_value,
                "delta": (new_value - old_value) if old_value is not None else None,
                "group": group_map.get(key),
                "score": score_map.get(key),
            }
        )
    updates_df = pd.DataFrame(rows)
    display(updates_df.sort_values("score", ascending=False))


## Constants Before/After

In [ ]:
if updates_df.empty:
    print("No proposed constant updates to visualize.")
else:
    viz_df = updates_df.copy().sort_values("delta")
    display(viz_df)

    y = np.arange(len(viz_df))
    h = 0.38
    plt.figure(figsize=(10, max(3, 0.6 * len(viz_df))))
    plt.barh(y - h/2, viz_df["old"], height=h, label="before", color="#a5a5a5")
    plt.barh(y + h/2, viz_df["new"], height=h, label="after (proposed)", color="#4472c4")
    plt.yticks(y, viz_df["constant"])
    plt.xlabel("constant value")
    plt.title("Engine Constant Proposals: Before vs After")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def format_value_like(old_raw: str, value: float) -> str:
    old_raw = old_raw.strip()
    if re.search(r"\.", old_raw):
        txt = f"{float(value):.4f}" if abs(float(value)) < 1 else f"{float(value):.3f}"
        txt = txt.rstrip("0").rstrip(".")
        return txt if txt else "0"
    return str(int(round(float(value))))


def apply_constant_updates(path: Path, updates: Dict[str, float]) -> Tuple[str, str, List[str]]:
    old_text = path.read_text(encoding="utf-8")
    changed_keys: List[str] = []

    new_lines: List[str] = []
    for line in old_text.splitlines(keepends=True):
        m = CONST_PATTERN.match(line.rstrip("\n"))
        if not m:
            new_lines.append(line)
            continue

        prefix, key, old_raw, suffix = m.groups()
        if key not in updates:
            new_lines.append(line)
            continue

        new_raw = format_value_like(old_raw, updates[key])
        rebuilt = f"{prefix}{new_raw}{suffix}\n"
        new_lines.append(rebuilt)
        changed_keys.append(key)

    new_text = "".join(new_lines)
    return old_text, new_text, sorted(set(changed_keys))


patch_applied = False
changed_constants: List[str] = []

slim_candidates = []
for c in candidate_results:
    slim_candidates.append({
        "name": c.get("name"),
        "scale": c.get("scale"),
        "proxy_score": c.get("proxy_score"),
        "estimated_cp_loss_reduction": c.get("estimated_cp_loss_reduction"),
        "changed_constants": c.get("changed_constants", []),
    })

model_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "summary": summary,
    "train_summary": train_summary_for_optimizer,
    "holdout_summary": holdout_summary_for_optimizer,
    "training_counts": training_counts,
    "model_metrics": model_metrics,
    "feature_importance": [{"feature": k, "importance": v} for k, v in feature_importance],
    "parameter_registry_path": str(PARAM_REGISTRY_PATH),
    "parameter_registry_size": len(optimization_payload.get("registry", {}).get("parameters", [])),
    "signal_context_train": signal_context_train,
    "signal_context_holdout": signal_context_holdout,
    "group_pressure_train": group_pressure_train,
    "group_pressure_holdout": group_pressure_holdout,
    "optimizer_training_volume_weight": optimizer_training_volume_weight,
    "parameter_weights": parameter_rows,
    "candidate_results": slim_candidates,
    "selected_candidate": {
        "name": selected_candidate.get("name"),
        "scale": selected_candidate.get("scale"),
        "proxy_score": selected_candidate.get("proxy_score"),
        "estimated_cp_loss_reduction": selected_candidate.get("estimated_cp_loss_reduction"),
        "changed_constants": selected_candidate.get("changed_constants", []),
    },
    "proposed_constant_updates": proposed_updates,
}
MODEL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_REPORT_PATH.write_text(json.dumps(model_report, indent=2), encoding="utf-8")
print(f"Wrote model report: {MODEL_REPORT_PATH}")

if proposed_updates:
    old_text, new_text, changed = apply_constant_updates(ENGINE_PM_PATH, proposed_updates)
    diff = "".join(
        difflib.unified_diff(
            old_text.splitlines(keepends=True),
            new_text.splitlines(keepends=True),
            fromfile=str(ENGINE_PM_PATH),
            tofile=str(ENGINE_PM_PATH),
        )
    )

    if WRITE_PATCH_PREVIEW:
        PATCH_PREVIEW_PATH.write_text(diff, encoding="utf-8")
        print(f"Wrote patch preview: {PATCH_PREVIEW_PATH}")

    changed_constants = changed
    if APPLY_ENGINE_PATCH:
        ENGINE_PM_PATH.write_text(new_text, encoding="utf-8")
        patch_applied = True
        print(f"Applied updates to {ENGINE_PM_PATH}")
        print("Changed constants:", ", ".join(changed_constants))
    else:
        print("APPLY_ENGINE_PATCH is False. No file changes applied.")
        print("Proposed changes:", ", ".join(changed_constants))
        if diff:
            print("\nPatch preview diff:\n")
            print(diff)
        print("Apply this migration with:")
        print(f"  {MIGRATION_CHECK_CMD}")
        print(f"  {MIGRATION_APPLY_CMD}")
else:
    print("No patch preview produced because no updates were proposed.")


## Post-Patch Verification (Optional)

Enable `RUN_POST_PATCH_EVAL=True` and `APPLY_ENGINE_PATCH=True` to run a true after-change comparison.

In [ ]:
summary_after: Dict[str, Any] | None = None

if RUN_POST_PATCH_EVAL and patch_applied:
    verify_games = games[:POST_PATCH_MAX_GAMES] if POST_PATCH_MAX_GAMES else games
    rows_after: List[Dict[str, Any]] = []
    verify_engine = UciEngineSession(ENGINE_CMD, REPO_ROOT)
    try:
        for i, game in enumerate(verify_games, start=1):
            game_rows = analyze_game(game, verify_engine)
            rows_after.extend(game_rows)
            print(f"Post-patch game {i}/{len(verify_games)} -> {len(game_rows)} rows")
    finally:
        verify_engine.close()

    df_after = pd.DataFrame(rows_after)
    summary_after = summarize_dataset(df_after)
    print("Post-patch summary:")
    print(json.dumps(summary_after, indent=2))

    metric_keys = ["bestmove_match_rate", "avg_cp_loss", "critical_80cp", "severe_150cp"]
    comp_rows = []
    for k in metric_keys:
        comp_rows.append({"metric": k, "before": summary.get(k), "after": summary_after.get(k)})
    comp_df = pd.DataFrame(comp_rows)
    display(comp_df)

    # Plot side-by-side where numeric values are available.
    plot_df = comp_df.dropna().copy()
    if len(plot_df):
        x = np.arange(len(plot_df))
        w = 0.38
        plt.figure(figsize=(9, 4))
        plt.bar(x - w/2, plot_df["before"], width=w, label="before", color="#a5a5a5")
        plt.bar(x + w/2, plot_df["after"], width=w, label="after", color="#70ad47")
        plt.xticks(x, plot_df["metric"], rotation=20)
        plt.title("Before vs After Patch Metrics")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Post-patch verification skipped. Set RUN_POST_PATCH_EVAL=True and APPLY_ENGINE_PATCH=True.")

In [ ]:
# Always write a compact run report for automation.
full_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pgn_path": str(ACTIVE_PGN_PATH),
    "engine_cmd": ENGINE_CMD,
    "input_source_mode": INPUT_SOURCE_MODE,
    "migration": {
        "dir": str(MIGRATION_DIR),
        "name": MIGRATION_DIR.name,
        "check_cmd": MIGRATION_CHECK_CMD,
        "apply_cmd": MIGRATION_APPLY_CMD,
    },
    "config": {
        "engine_name": ENGINE_NAME,
        "allow_generic_pgn_games": ALLOW_GENERIC_PGN_GAMES,
        "generic_side_mode": GENERIC_ANALYZE_SIDE_MODE,
        "max_ply_per_game": MAX_PLY_PER_GAME,
        "max_games": MAX_GAMES,
        "opening_ply_limit": OPENING_PLY_LIMIT,
        "search_movetime_ms": SEARCH_MOVETIME_MS,
        "eval_movetime_ms": EVAL_MOVETIME_MS,
        "analyze_played_eval": ANALYZE_PLAYED_EVAL,
        "enable_book_compare": ENABLE_BOOK_COMPARE,
        "min_rows_for_model": MIN_ROWS_FOR_MODEL,
        "severe_cp_loss": SEVERE_CP_LOSS,
        "holdout_fraction": HOLDOUT_FRACTION,
        "candidate_scales": CANDIDATE_SCALES,
        "max_constant_updates": MAX_CONSTANT_UPDATES,
        "min_param_score_for_update": MIN_PARAM_SCORE_FOR_UPDATE,
        "training_reference_games": TRAINING_REFERENCE_GAMES,
        "training_volume_exponent": TRAINING_VOLUME_EXPONENT,
    },
    "summary_before": summary,
    "summary_after": summary_after,
    "train_summary": train_summary_for_optimizer,
    "holdout_summary": holdout_summary_for_optimizer,
    "training_counts": training_counts,
    "model_metrics": model_metrics,
    "feature_importance": [{"feature": k, "importance": v} for k, v in feature_importance],
    "parameter_registry_path": str(PARAM_REGISTRY_PATH),
    "signal_context_train": signal_context_train,
    "signal_context_holdout": signal_context_holdout,
    "group_pressure_train": group_pressure_train,
    "group_pressure_holdout": group_pressure_holdout,
    "optimizer_training_volume_weight": optimizer_training_volume_weight,
    "selected_candidate": {
        "name": selected_candidate.get("name"),
        "scale": selected_candidate.get("scale"),
        "proxy_score": selected_candidate.get("proxy_score"),
        "estimated_cp_loss_reduction": selected_candidate.get("estimated_cp_loss_reduction"),
    },
    "proposed_constant_updates": proposed_updates,
    "applied_patch": bool(patch_applied),
    "changed_constants": changed_constants,
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(full_report, indent=2), encoding="utf-8")
print(f"Wrote run report: {REPORT_PATH}")
